# MP25 — Combined Attention Module

Sequential combined attention first weights channels, then computes a spatial gate from the already gated features. This implementation is CBAM-style, not a faithful CBAM reproduction: its channel branch uses average pooling only.

## Mathematics and worked example

F1=A_channel(F)⊙F; output=A_spatial(F1)⊙F1. The two gates broadcast over different axes. With scalar gates 0.5 and 0.25, a value 8 becomes 1. However swapping the modules generally changes the learned input-dependent gates; multiplication of fixed scalars commuting does not prove module order is irrelevant. The width-8 gate overhead is 42+99=141 parameters.

In [ ]:
from pathlib import Path
import sys,json,torch
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from rssr.attention_labs import InspectableAttention
torch.manual_seed(23)
x=torch.rand(2,8,5,7)
g=InspectableAttention(8)
y,c,s=g(x,'combined')
torch.testing.assert_close(y,x*c*s)
assert sum(p.numel() for p in g.parameters())==141
for mode in ['off','uniform','channel','spatial','combined']:
 print(mode,g(x,mode)[0].norm().item())

## Executed experiment and reproducibility

Run `python scripts/attention_lessons_experiment.py` from the repository root. This is same-grid RGB reconstruction, not super-resolution. A tiny shape fixture precedes training on four NAIP scenes, with four separate validation scenes, 16 fixed 32×32 patches each. USDA/OpenSR NAIP imagery is public domain; values are stored DN/10000, not claimed calibrated reflectance. No new imagery download or final-test access is required.

CPU float32, two threads, seed 23, Adam 0.001, batch four, 300 updates, MSE. Models share identical head/tail initialization and minibatch sequence. Evaluate the fixed last step, with no checkpoint selection. Validation PSNR uses range one, border two, equal-size patches and equal scene weighting. Inactive gate modules remain allocated to preserve initialization; the report separately counts active and allocated parameters. Inference timing uses five warmups and twenty observations on a 32×32 input; it is not whole-app latency.

The first figure compares separately trained reconstructions under one reference contrast scale. The second applies alternative gate modes to identical features from the trained combined model, plus synthetic step-edge/checkerboard fixtures. All mask heatmaps share [0,1]; these are not semantic labels. Training curves show sampled minibatch loss, not a full-dataset estimate. Read the actual measured table below; preserve unfavorable outcomes.


In [ ]:
r=json.loads((ROOT/'reports/mp23-mp25-attention.json').read_text())
for row in r['results']:
 print({k:v for k,v in row.items() if k!='per_tile'})
from IPython.display import display,Image
for name in ['mp23-mp25-reconstruction.png','mp23-mp25-attention.png']:
 display(Image(filename=str(ROOT/'reports'/name)))

## Practice

1. Why inspect modes on the same learned features?
2. Do fixed gates imply the modules commute?
3. Does a lower feature norm mean better reconstruction?

## Worked solutions

1. It isolates gating effects from different upstream feature extractors; separately trained models answer a different question.
2. No. The second module recomputes a gate from the first module’s output.
3. No. Evaluate output error against a reference, and keep the metric protocol fixed.

## Limitations

One seed and a short reconstruction experiment cannot establish SR superiority, causality, explainability or geographic generalization. Equal update count is not equal computation. Learner mastery is unassessed. Frozen research weights and test results are untouched.

[Authoritative brief](../stage_06_attention/MP25_combined_attention_module/README.md) · [Local/Colab setup](../README.md)